# Система інтелектуального збагачення Excel-даних

Система приймає Excel-файл і текстовий опис завдання, сама визначає, яку колонку треба заповнити, шукає дані в інтернеті та зберігає новий файл.

Граф складається з чотирьох вузлів: `read_file` → `plan_task` → `enrich_rows` → `save_file`.
Для рядків, де звичайний пошук нічого не дав, підключається агент-дослідник з інструментом пошуку.

### Встановлення пакетів

In [162]:
!pip install -q -U "langgraph>=1.0" langchain langchain-openai langchain-core tavily-python pandas openpyxl
import warnings
warnings.filterwarnings("ignore")

### Налаштування ключів

In [163]:
import os
from getpass import getpass

def get_secret(name: str) -> str:
    # 1) змінна середовища, 2) Colab Secrets, 3) ручне введення
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass(f"Введіть {name}: ")

os.environ["OPENROUTER_API_KEY"] = get_secret("OPENROUTER_API_KEY")
os.environ["TAVILY_API_KEY"] = get_secret("TAVILY_API_KEY")

### Ініціалізація моделі та пошуку

Модель `gpt-4o-mini` через OpenRouter, пошук через Tavily. Для моделі задано `max_tokens`, бо без нього OpenRouter резервує під кожну відповідь максимальний обсяг і на малому балансі запит відхиляється.

In [164]:
import re
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from typing import TypedDict, Literal, Optional, List, Any

import pandas as pd
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.graph import StateGraph, START, END
from tavily import TavilyClient

llm = ChatOpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-4o-mini",
    temperature=0,
    max_tokens=1000   # відповіді короткі, великий ліміт не потрібен
)

tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

MAX_WORKERS = 5      # скільки рядків обробляється одночасно
MAX_RETRIES = 3      # скільки разів повторюємо запит при помилці
SEARCH_RESULTS = 5   # скільки результатів пошуку передаємо моделі
SAVE_EVERY = 50      # після скількох запитів зберігаємо проміжний результат
AGENT_STEPS = 10     # обмеження кроків агента-дослідника (до 4 пошуків)

### Типи даних

In [165]:
class EnrichmentPlan(TypedDict):
    """План збагачення таблиці"""
    target_column: str
    needs_search: bool
    search_query_template: str
    value_type: Literal["number", "text"]
    unit: str

class ExtractedValue(TypedDict):
    """Значення для однієї клітинки"""
    found: bool
    value: str

class EnrichmentState(TypedDict):
    """Стан графа"""
    file_path: str
    task_description: str
    output_path: Optional[str]
    df: Optional[Any]
    plan: Optional[EnrichmentPlan]
    source_columns: List[str]
    filled: int
    errors: List[str]
    fatal_error: Optional[str]

### Допоміжні функції

`call_with_retry` повторює запит, якщо сервіс тимчасово не відповів. Якщо перевищено ліміт запитів, пауза перед повтором довша. `to_number` дістає число з відповіді моделі.

In [166]:
def call_with_retry(func, *args, **kwargs):
    """Виклик функції з повторними спробами"""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            # при перевищенні ліміту запитів чекаємо довше
            text = f"{type(e).__name__} {e}".lower()
            rate_limited = "429" in text or "limit" in text
            time.sleep(10 * attempt if rate_limited else attempt)

def is_empty(value) -> bool:
    return pd.isna(value) or str(value).strip() == ""

def to_number(text: str):
    """'8,849 m' -> 8849, '2023.5' -> 2023.5"""
    cleaned = text.replace(" ", "").replace("\u00a0", "")
    cleaned = re.sub(r",(?=\d{3}(\D|$))", "", cleaned)   # кома як роздільник тисяч
    cleaned = cleaned.replace(",", ".")
    match = re.search(r"-?\d+(\.\d+)?", cleaned)
    if not match:
        return None
    number = float(match.group())
    return int(number) if number.is_integer() else round(number, 2)

def build_query(template: str, row: dict) -> str:
    """Підставляє значення з рядка у шаблон запиту"""
    query = template
    for column, value in row.items():
        query = query.replace("{" + str(column) + "}", str(value))
    return query

### Вузол 1: читання файлу

In [167]:
def read_file(state: EnrichmentState) -> dict:
    path = state["file_path"]

    if not state["task_description"].strip():
        return {"fatal_error": "Опис завдання порожній"}
    if not os.path.exists(path):
        return {"fatal_error": f"Файл '{path}' не знайдено"}

    # результат завжди пишемо в окремий файл, оригінал не чіпаємо
    source = Path(path)
    output_path = state.get("output_path") or str(source.with_name(f"{source.stem}_enriched.xlsx"))
    if os.path.abspath(output_path) == os.path.abspath(path):
        return {"fatal_error": "Файл результату не може збігатися з вхідним файлом"}

    try:
        df = pd.read_excel(path)
    except Exception as e:
        return {"fatal_error": f"Не вдалося прочитати файл '{path}': {e}"}

    if df.empty:
        return {"fatal_error": f"У файлі '{path}' немає рядків з даними"}

    print(f"📄 Прочитано '{path}': {len(df)} рядків, колонки: {list(df.columns)}")
    return {"df": df, "output_path": output_path}

### Вузол 2: аналіз завдання

Модель бачить назви колонок, перші рядки та опис завдання і повертає план: яку колонку заповнювати, чи потрібен пошук, шаблон пошукового запиту і тип значення.
Це один запит до моделі на весь файл. Далі для кожного рядка запит будується з шаблону вже без моделі.

In [168]:
def plan_task(state: EnrichmentState) -> dict:
    df = state["df"]
    columns = [str(c) for c in df.columns]

    plan_prompt = f"""
        Ти аналізуєш таблицю і завдання користувача та складаєш план, як доповнити таблицю.

        Колонки таблиці: {columns}
        Перші рядки: {df.head(3).to_string(index=False)}

        Завдання: {state['task_description']}

        Правила:
        - target_column — назва колонки, яку треба заповнити. Якщо така колонка вже є в таблиці,
          поверни її назву точно як у таблиці. Якщо немає — дай коротку назву для нової колонки.
        - needs_search — true, якщо значення є фактом, який треба шукати в інтернеті
          (відстані, висоти, населення, дати тощо). false — тільки якщо значення можна отримати
          із самих даних рядка (обчислення, переклад, форматування).
        - search_query_template — шаблон пошукового запиту англійською мовою для одного рядка.
          Значення з рядка підставляй через назви колонок у фігурних дужках, наприклад {{{columns[0]}}}.
          Використовуй тільки колонки з таблиці, цільову колонку не використовуй.
          Запит має бути точним: що саме шукаємо і в яких одиницях
          (наприклад, пряма відстань — це straight line distance, а не відстань дорогами).
        - value_type — number, якщо результат є числом, інакше text.
        - unit — одиниця виміру англійською (km, m тощо) або порожній рядок.
        """

    try:
        plan = call_with_retry(llm.with_structured_output(EnrichmentPlan).invoke, plan_prompt)
    except Exception as e:
        return {"fatal_error": f"Не вдалося проаналізувати завдання: {e}"}

    # цільова колонка: шукаємо без урахування регістру, якщо немає — створюємо
    target = str(plan.get("target_column", "")).strip()
    if not target:
        return {"fatal_error": "Не вдалося визначити, яку колонку заповнювати"}
    for column in df.columns:
        if str(column).lower() == target.lower():
            target = column
            break
    else:
        df[target] = None
        print(f"➕ Додано нову колонку '{target}'")
    plan["target_column"] = target

    # перевіряємо, що шаблон посилається тільки на існуючі колонки
    template = plan.get("search_query_template", "")
    used = re.findall(r"\{([^{}]+)\}", template)
    source_columns = [c for c in df.columns if str(c) in used and c != target]
    if not source_columns or len(source_columns) != len(set(used)):
        source_columns = [c for c in df.columns if c != target]
        template = " ".join("{" + str(c) + "}" for c in source_columns) + " " + state["task_description"]
        plan["search_query_template"] = template
        print("⚠️ Шаблон запиту від моделі некоректний, використовую запит з усіх колонок")

    print(f"🧭 Колонка: '{target}', тип: {plan['value_type']}, пошук: {plan['needs_search']}")
    print(f"   Шаблон запиту: {template}")
    return {"df": df, "plan": plan, "source_columns": source_columns}

### Пошук і агент-дослідник

`search_web` виконує один пошук і повертає текст результатів.
Агент-дослідник використовується тільки як запасний варіант: він отримує той самий пошук як інструмент і може сам переформулювати запит кілька разів.

In [169]:
def search_web(query: str) -> str:
    """Один пошук у Tavily. Повертає текст результатів або порожній рядок"""
    response = call_with_retry(
        tavily.search, query, max_results=SEARCH_RESULTS, include_answer=True
    )
    parts = []
    if response.get("answer"):
        parts.append(f"Коротка відповідь пошуку: {response['answer']}")
    for item in response.get("results", []):
        parts.append(f"- {item.get('title', '')}: {item.get('content', '')}")
    return "\n".join(parts)


@tool
def web_search(query: str) -> str:
    """Пошук інформації в інтернеті. query - пошуковий запит англійською мовою."""
    try:
        return search_web(query) or "Нічого не знайдено"
    except Exception as e:
        return f"Помилка пошуку: {e}"


research_prompt = """Ти - дослідник, який шукає одне конкретне значення в інтернеті.

Використовуй інструмент web_search. Якщо пошук не дав відповіді, сформулюй запит інакше
(інша назва, уточнення, інші слова), але зроби не більше трьох пошуків.
Відповідай тільки на основі результатів пошуку, не використовуй власні знання і нічого не вигадуй.
У відповіді напиши знайдене значення з одиницями виміру і коротко, звідки воно.
Якщо значення не знайдено, напиши: NOT FOUND."""

research_agent = create_agent(llm, tools=[web_search], system_prompt=research_prompt)

### Вузол 3: пошук і заповнення

Для кожного рядка з порожньою цільовою колонкою будується запит. Однакові запити шукаються один раз, запити виконуються паралельно через `ThreadPoolExecutor`.

Значення шукається у два кроки:
1. пошук за шаблоном і один запит до моделі, яка дістає значення з результатів;
2. якщо значення не знайдено — агент-дослідник пробує інші формулювання.

Після кожних `SAVE_EVERY` запитів проміжний результат зберігається у файл. Якщо обробка обірветься, можна запустити функцію на збереженому файлі, і вона заповнить тільки порожні клітинки.
Якщо значення так і не знайдено або сервіс повернув помилку, клітинка залишається порожньою, а причина потрапляє у список помилок.

In [170]:
def extract_value(query: str, data_title: str, data: str, plan: EnrichmentPlan, task_description: str):
    """Дістає значення з наведених даних. Повертає (значення, текст помилки)"""
    extract_prompt = f"""
    Завдання користувача: {task_description}
    Запит: {query}
    Потрібне значення для колонки "{plan['target_column']}"
    Тип: {plan['value_type']}, одиниця виміру: {plan.get('unit') or 'немає'}

    {data_title}:{data}

    Правила:
    - found = true тільки якщо значення можна надійно визначити з наведених даних. Нічого не вигадуй.
    - якщо джерела дають різні числа, обери те, яке точно відповідає завданню і зустрічається найчастіше
    - якщо значення в інших одиницях (милі, фути), переведи в потрібні
    - для числа: у value тільки число без одиниць і пробілів, дробова частина через крапку
    - для тексту: у value коротка відповідь без пояснень
    - якщо значення не знайдено: found = false, value = ""
    """

    try:
        answer = call_with_retry(llm.with_structured_output(ExtractedValue).invoke, extract_prompt)
    except Exception as e:
        return None, f"помилка моделі: {e}"

    value = str(answer.get("value", "")).strip()
    if not answer.get("found") or not value:
        return None, "значення не знайдено"

    if plan["value_type"] == "number":
        number = to_number(value)
        if number is None:
            return None, f"не вдалося отримати число з відповіді '{value}'"
        return number, None
    return value, None


def find_value(query: str, row: dict, plan: EnrichmentPlan, task_description: str):
    """Шукає значення для одного запиту. Повертає (значення, текст помилки)"""
    if not plan["needs_search"]:
        return extract_value(query, "Дані рядка", str(row), plan, task_description)

    # крок 1: пошук за шаблоном
    try:
        data = search_web(query)
    except Exception as e:
        return None, f"помилка пошуку: {e}"

    if data:
        value, error = extract_value(query, "Результати пошуку", data, plan, task_description)
        # якщо значення знайдено або сервіс недоступний, агент не потрібен
        if error is None or error.startswith("помилка"):
            return value, error

    # крок 2: агент-дослідник пробує інші формулювання
    print(f"   🔁 повторний пошук агентом: {query}")
    question = f"""Завдання: {task_description}
        Дані рядка: {row}
        Запит, який не дав результату: {query}
        Знайди значення для колонки "{plan['target_column']}" (одиниця виміру: {plan.get('unit') or 'немає'})."""

    try:
        result = research_agent.invoke(
            {"messages": [{"role": "user", "content": question}]},
            {"recursion_limit": AGENT_STEPS}
        )
        answer = str(result["messages"][-1].content)
    except Exception as e:
        return None, f"значення не знайдено, повторний пошук не вдався: {e}"

    return extract_value(query, "Відповідь дослідника", answer, plan, task_description)


def enrich_rows(state: EnrichmentState) -> dict:
    df = state["df"]
    plan = state["plan"]
    target = plan["target_column"]
    task = state["task_description"]
    errors = []

    # запит для кожного рядка, де цільова колонка порожня; однакові запити об'єднуються
    query_rows = {}   # запит -> номери рядків
    query_data = {}   # запит -> дані рядка
    for index, row in df.iterrows():
        if not is_empty(row[target]):
            continue
        if any(is_empty(row[c]) for c in state["source_columns"]):
            errors.append(f"Рядок {index + 2}: не вистачає вхідних даних, пропущено")
            continue
        row_data = {c: row[c] for c in state["source_columns"]}
        query = build_query(plan["search_query_template"], row_data)
        query_rows.setdefault(query, []).append(index)
        query_data[query] = row_data

    queries = list(query_rows)
    rows_count = sum(len(indexes) for indexes in query_rows.values())
    print(f"🔍 Рядків для заповнення: {rows_count}, унікальних запитів: {len(queries)}")

    df[target] = df[target].astype(object)
    filled = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for start in range(0, len(queries), SAVE_EVERY):
            batch = queries[start:start + SAVE_EVERY]
            results = list(executor.map(lambda q: find_value(q, query_data[q], plan, task), batch))

            # записуємо знайдені значення, інші колонки не чіпаємо
            service_errors = 0
            for query, (value, error) in zip(batch, results):
                for index in query_rows[query]:
                    if error:
                        errors.append(f"Рядок {index + 2} ({query}): {error}")
                    else:
                        df.at[index, target] = value
                        filled += 1
                if error and error.startswith("помилка"):
                    service_errors += 1

            done = start + len(batch)
            if done < len(queries):
                # проміжне збереження
                try:
                    df.to_excel(state["output_path"], index=False)
                    print(f"   оброблено {done}/{len(queries)}, проміжний результат збережено")
                except Exception as e:
                    print(f"⚠️ Проміжне збереження не вдалося: {e}")

                # якщо жоден запит не пройшов, сервіс недоступний і далі чекати немає сенсу
                if service_errors == len(batch):
                    errors.append(f"Обробку зупинено: сервіс не відповідає, не оброблено запитів: {len(queries) - done}")
                    break

    return {"df": df, "filled": filled, "errors": errors}

### Вузол 4: збереження файлу

In [171]:
def save_file(state: EnrichmentState) -> dict:
    try:
        state["df"].to_excel(state["output_path"], index=False)
    except Exception as e:
        return {"fatal_error": f"Не вдалося зберегти файл '{state['output_path']}': {e}"}

    return {"output_path": state["output_path"]}

### Побудова графа

Після читання файлу та після аналізу завдання є перевірка: якщо сталася помилка, граф одразу завершується.

In [172]:
def check_error(state: EnrichmentState) -> str:
    return "stop" if state.get("fatal_error") else "continue"

def build_enrichment_agent():
    workflow = StateGraph(EnrichmentState)

    workflow.add_node("read_file", read_file)
    workflow.add_node("plan_task", plan_task)
    workflow.add_node("enrich_rows", enrich_rows)
    workflow.add_node("save_file", save_file)

    workflow.add_edge(START, "read_file")
    workflow.add_conditional_edges("read_file", check_error, {"continue": "plan_task", "stop": END})
    workflow.add_conditional_edges("plan_task", check_error, {"continue": "enrich_rows", "stop": END})
    workflow.add_edge("enrich_rows", "save_file")
    workflow.add_edge("save_file", END)

    return workflow.compile()

agent = build_enrichment_agent()
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	read_file(read_file)
	plan_task(plan_task)
	enrich_rows(enrich_rows)
	save_file(save_file)
	__end__([<p>__end__</p>]):::last
	__start__ --> read_file;
	enrich_rows --> save_file;
	plan_task -. &nbsp;stop&nbsp; .-> __end__;
	plan_task -. &nbsp;continue&nbsp; .-> enrich_rows;
	read_file -. &nbsp;stop&nbsp; .-> __end__;
	read_file -. &nbsp;continue&nbsp; .-> plan_task;
	save_file --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Функція запуску

In [173]:
def process_excel(file_path: str, task_description: str, output_path: Optional[str] = None):
    """Збагачує Excel-файл за текстовим описом завдання.

    file_path - шлях до вхідного файлу
    task_description - що треба знайти і в яку колонку записати
    output_path - куди зберегти результат (за замовчуванням <назва>_enriched.xlsx)

    Повертає шлях до створеного файлу або None, якщо сталася помилка.
    """
    print("=" * 70)
    print(f"ЗАВДАННЯ: {task_description}")
    print("=" * 70)
    start_time = time.time()

    result = agent.invoke({
        "file_path": file_path,
        "task_description": task_description,
        "output_path": output_path,
        "source_columns": [],
        "filled": 0,
        "errors": [],
        "fatal_error": None
    })

    if result.get("fatal_error"):
        print(f"❌ ПОМИЛКА: {result['fatal_error']}")
        return None

    for error in result["errors"]:
        print(f"⚠️ {error}")

    target = result["plan"]["target_column"]
    empty_left = int(result["df"][target].map(is_empty).sum())
    print(f"✅ Заповнено клітинок: {result['filled']}, залишилось порожніх: {empty_left}")
    print(f"💾 Створено файл {result['output_path']} ({time.time() - start_time:.1f} с)")
    return result["output_path"]

### Тестування

Тестові файли `capitals.xlsx` і `mountains.xlsx` завантажуються з репозиторію на GitHub.

In [174]:
import urllib.request

REPO_URL = "https://github.com/valeriandema/Agentic-AI/raw/refs/heads/main/"

for name in ["capitals.xlsx", "mountains.xlsx"]:
    try:
        urllib.request.urlretrieve(REPO_URL + name, name)
        print(f"⬇️ Завантажено {name}")
    except Exception as e:
        print(f"❌ Не вдалося завантажити {name}: {e}")

⬇️ Завантажено capitals.xlsx
⬇️ Завантажено mountains.xlsx


Столиці

In [175]:
capitals_path = process_excel(
    file_path="capitals.xlsx",
    task_description="знайди пряму відстань між столицями в км для колонки distance",
    output_path="capitals_enriched.xlsx"
)
pd.read_excel(capitals_path) if capitals_path else None

ЗАВДАННЯ: знайди пряму відстань між столицями в км для колонки distance
📄 Прочитано 'capitals.xlsx': 10 рядків, колонки: ['Capital_From', 'Country_From', 'Capital_To', 'Country_To', 'distance']
🧭 Колонка: 'distance', тип: number, пошук: True
   Шаблон запиту: straight line distance between {Capital_From} and {Capital_To} in km
🔍 Рядків для заповнення: 10, унікальних запитів: 10
✅ Заповнено клітинок: 10, залишилось порожніх: 0
💾 Створено файл capitals_enriched.xlsx (9.6 с)


,Capital_From,Country_From,Capital_To,Country_To,distance
0,Kyiv,Ukraine,Paris,France,2032.00
1,London,United Kingdom,Rome,Italy,1435.00
2,Paris,France,Berlin,Germany,878.00
3,Berlin,Germany,Vienna,Austria,523.00
4,Warsaw,Poland,Kyiv,Ukraine,410.00
5,Rome,Italy,Madrid,Spain,1365.00
6,Madrid,Spain,Lisbon,Portugal,503.00
7,Vienna,Austria,Budapest,Hungary,214.17
8,Stockholm,Sweden,Oslo,Norway,416.00
9,Athens,Greece,Istanbul,Turkey,562.00


Гори

In [176]:
mountains_path = process_excel(
    file_path="mountains.xlsx",
    task_description="додай висоту гір у метрах до колонки height",
    output_path="mountains_enriched.xlsx"
)
pd.read_excel(mountains_path) if mountains_path else None

ЗАВДАННЯ: додай висоту гір у метрах до колонки height
📄 Прочитано 'mountains.xlsx': 10 рядків, колонки: ['Mountain', 'Country', 'height']
🧭 Колонка: 'height', тип: number, пошук: True
   Шаблон запиту: What is the height of {Mountain} in meters?
🔍 Рядків для заповнення: 10, унікальних запитів: 10
✅ Заповнено клітинок: 10, залишилось порожніх: 0
💾 Створено файл mountains_enriched.xlsx (7.4 с)


,Mountain,Country,height
0,Mount Everest,Nepal/China,8848.86
1,K2,Pakistan/China,8611.00
2,Kangchenjunga,Nepal/India,8586.00
3,Lhotse,Nepal/China,8516.00
4,Makalu,Nepal/China,8463.00
5,Cho Oyu,Nepal/China,8201.00
6,Dhaulagiri,Nepal,8167.00
7,Manaslu,Nepal,8163.00
8,Nanga Parbat,Pakistan,8126.00
9,Annapurna,Nepal,8091.00


### Перевірка обробки помилок

Файл, якого не існує:

In [177]:
process_excel("no_such_file.xlsx", "додай висоту гір у метрах до колонки height")

ЗАВДАННЯ: додай висоту гір у метрах до колонки height
❌ ПОМИЛКА: Файл 'no_such_file.xlsx' не знайдено


Таблиця, де одна гора справжня, а другої не існує. Для неіснуючої гори має спрацювати повторний пошук агентом, а клітинка має залишитися порожньою.

In [178]:
pd.DataFrame({
    "Mountain": ["Hoverla", "Qwertyzzz Peak"],
    "Country": ["Ukraine", "Nowhereland"],
    "height": [None, None]
}).to_excel("test_unknown.xlsx", index=False)

test_path = process_excel("test_unknown.xlsx", "додай висоту гір у метрах до колонки height")
pd.read_excel(test_path) if test_path else None

ЗАВДАННЯ: додай висоту гір у метрах до колонки height
📄 Прочитано 'test_unknown.xlsx': 2 рядків, колонки: ['Mountain', 'Country', 'height']
🧭 Колонка: 'height', тип: number, пошук: True
   Шаблон запиту: What is the height of {Mountain} in meters?
🔍 Рядків для заповнення: 2, унікальних запитів: 2
   🔁 повторний пошук агентом: What is the height of Qwertyzzz Peak in meters?
⚠️ Рядок 3 (What is the height of Qwertyzzz Peak in meters?): значення не знайдено
✅ Заповнено клітинок: 1, залишилось порожніх: 1
💾 Створено файл test_unknown_enriched.xlsx (13.6 с)


,Mountain,Country,height
0,Hoverla,Ukraine,2061.0
1,Qwertyzzz Peak,Nowhereland,NaN


### Висновок

Система працює з обома файлами без змін у коді: назву колонки, шаблон пошукового запиту і тип значення модель визначає з опису завдання. На весь файл іде один запит до моделі для плану, а далі на кожен унікальний рядок один пошук і один запит до моделі. Агент-дослідник підключається тільки для рядків, де значення не знайшлося з першого разу, тому зайвих запитів майже немає.

Рядки обробляються паралельно, проміжний результат зберігається кожні 50 запитів. Оригінальні дані не змінюються — заповнюються тільки порожні клітинки цільової колонки, а якщо дані не знайдено, клітинка залишається порожньою і виводиться причина.